In [2]:
from IPython.display import clear_output
! pip install alphagenome
clear_output()

In [2]:
import numpy  as np
import pandas as pd
import os
import argparse

import time
import random
# import grpc
# from grpc import StatusCode

from alphagenome.data import genome
from alphagenome.models import dna_client, variant_scorers

/hpc/group/igvf/revathy/software/miniconda3/envs/alphagenome/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
os.environ["ALPHAGENOME_API_KEY"] = "xxxxxxx"
dna_model = dna_client.create(os.environ["ALPHAGENOME_API_KEY"])

In [38]:
output_metadata = dna_model.output_metadata(
    dna_client.Organism.HOMO_SAPIENS
).concatenate()

In [40]:
output_metadata[output_metadata['biosample_name']=='A549']

,name,strand,Assay title,ontology_curie,biosample_name,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,nonzero_mean,output_type,gtex_tissue,histone_mark,transcription_factor
55,EFO:0001086 ATAC-seq,.,ATAC-seq,EFO:0001086,A549,cell_line,adult,encode,single,False,1.497697,OutputType.ATAC,NaN,NaN,NaN
144,hCAGE EFO:0001086,+,hCAGE,EFO:0001086,A549,cell_line,NaN,fantom,NaN,NaN,98.964256,OutputType.CAGE,NaN,NaN,NaN
417,hCAGE EFO:0001086,-,hCAGE,EFO:0001086,A549,cell_line,NaN,fantom,NaN,NaN,98.964256,OutputType.CAGE,NaN,NaN,NaN
107,EFO:0001086 DNase-seq,.,DNase-seq,EFO:0001086,A549,cell_line,unknown,encode,single,False,1.369808,OutputType.DNASE,NaN,NaN,NaN
107,EFO:0001086 polyA plus RNA-seq,+,polyA plus RNA-seq,EFO:0001086,A549,cell_line,adult,encode,paired,False,0.129406,OutputType.RNA_SEQ,,NaN,NaN
378,EFO:0001086 polyA plus RNA-seq,-,polyA plus RNA-seq,EFO:0001086,A549,cell_line,adult,encode,paired,False,0.129406,OutputType.RNA_SEQ,,NaN,NaN
328,EFO:0001086 Histone ChIP-seq H3K27ac,.,Histone ChIP-seq,EFO:0001086,A549,cell_line,adult,encode,single,False,0.632995,OutputType.CHIP_HISTONE,NaN,H3K27ac,NaN
329,EFO:0001086 Histone ChIP-seq H3K4me1,.,Histone ChIP-seq,EFO:0001086,A549,cell_line,adult,encode,single,False,0.688574,OutputType.CHIP_HISTONE,NaN,H3K4me1,NaN
330,EFO:0001086 Histone ChIP-seq H3K4me2,.,Histone ChIP-seq,EFO:0001086,A549,cell_line,adult,encode,single,False,0.671028,OutputType.CHIP_HISTONE,NaN,H3K4me2,NaN
331,EFO:0001086 Histone ChIP-seq H3K4me3,.,Histone ChIP-seq,EFO:0001086,A549,cell_line,adult,encode,single,False,0.675506,OutputType.CHIP_HISTONE,NaN,H3K4me3,NaN


In [18]:
interval = genome.Interval('chr2', 11_133_551, 11_146_950).resize(
    dna_client.SEQUENCE_LENGTH_16KB)


In [19]:
interval

Interval(chromosome='chr2', start=11132059, end=11148443, strand='.', name='')

In [35]:
output = dna_model.predict_interval(
    interval=interval,
    requested_outputs={
        dna_client.OutputType.ATAC
    },
    ontology_terms=[],
)

In [36]:
atac_profiles = output.atac
[f for f in dir(atac_profiles) if not f.startswith('_')]

['bin_index',
 'change_resolution',
 'copy',
 'downsample',
 'filter_to_negative_strand',
 'filter_to_nonnegative_strand',
 'filter_to_nonpositive_strand',
 'filter_to_positive_strand',
 'filter_to_stranded',
 'filter_to_unstranded',
 'filter_tracks',
 'groupby',
 'interval',
 'metadata',
 'names',
 'num_tracks',
 'ontology_terms',
 'pad',
 'positional_axes',
 'resize',
 'resolution',
 'reverse_complement',
 'select_tracks_by_index',
 'select_tracks_by_name',
 'slice_by_interval',
 'slice_by_positions',
 'strands',
 'uns',
 'upsample',
 'values',
 'width']

In [37]:
atac_profiles.values

array([[7.81250000e-02, 6.93359375e-02, 7.22656250e-02, ...,
        6.88476562e-02, 5.83496094e-02, 8.88671875e-02],
       [2.90527344e-02, 2.86865234e-02, 2.84423828e-02, ...,
        2.90527344e-02, 2.45361328e-02, 3.61328125e-02],
       [1.04003906e-01, 9.76562500e-02, 9.22851562e-02, ...,
        1.01074219e-01, 8.20312500e-02, 1.21582031e-01],
       ...,
       [6.63101673e-07, 1.36345625e-06, 1.23679638e-06, ...,
        1.17719173e-06, 8.04662704e-06, 3.99351120e-06],
       [4.93526459e-05, 5.07831573e-05, 5.65052032e-05, ...,
        2.09808350e-04, 4.36782837e-04, 3.35693359e-04],
       [2.51770020e-03, 2.38037109e-03, 1.86920166e-03, ...,
        9.52148438e-03, 9.64355469e-03, 9.46044922e-03]],
      shape=(16384, 167), dtype=float32)

In [34]:
import pandas as pd

sequence = pd.read_csv('/hpc/home/rv103/igvf/revathy/GR-AP1/alphagenome/data/original-seq-16kb.txt', sep='\t', names=['sequence'])
seq_list = list(sequence['sequence'])
seq_list[0]

'CTCCAGCCTGGGTGACAGAGCAAGACTCCATCTCAAAAGAAAAAAAAAAAAAGAAAAAAAGAAAGAAACATGGAATATGTCTAACAACATTCCTCTCATTTTTCAGGGAAGGAGACAGAAGCTGAAAAGGTGAAGTGATAAGAGTCAGATCTAGAACCTAGTTTCCCCATCTCTAAGCCAAGGCCACAAAGTTCAGTGTGGATGGATTCTATTTTAGTTTACTCTTCAGTGGTTTATCTTGTTCCCTATCCTAATCTACAACTTGCCTCATGTTCTTTCCTAAGAAGTGGGGTTCAAATCTTAAGCAAATAATTATTAAATGCTGGACTGATCTGGCTGCCTTTTAAAACAGAGAGCAGAGATAAAGTACTTTAATTACAAGGATCAAGAATTGCAAAGCTCTTGGACAAATGAGCAAAGGCAATTCAATGGAGAAAGGATGGTCTTTTCAACAAATGATGCTGGAAAAATTGGACGTCCACAATTAACTTAGGCACAGACCTTAAACCTCTTACAAAAGTTATCTCAAAATGGATCATAGACCATTACATTTTAAACATAAAGTGCAAAACTGAAAATTCTAGAAGAAAACACGGCAGAAAAATCGATGCGATCTTGGATTTGGTGATGAATCTTTAGATATAATACAACATCAAAAGCACATTCTACGAAAGAATTTCAACCTGTGTTGTCCTCCACAATTATATCCACGTGTGTGCACAGACCTACCCTTCAGTCTATGGGAACAGTGTAGAGTGGGGAAAGGGCAGAGACTGTCCGGCCAAGCCAAAAGACTTGGGTTCTAGTCAGAGGCCTGCACCTCGTTAGCTGTGTCCCCTCAGGCAAGTCACTTTTCCAGATTCCAATTTTCTCATAGGTAAAATGAGTATATAATAATTCTAACCTAGCTCAGGTGGTTCTTAAAAGAGAAAAAGAAAGTAACATACATGAAAATATTTACAAGACAATGGTTATAATTTTGGTATATATAAAAATCAT

In [22]:
output = dna_model.predict_sequences(
    sequences=seq_list,
    requested_outputs={
        dna_client.OutputType.ATAC
    },
    ontology_terms=['EFO:0001086'],
)

100%|██████████| 12/12 [00:00<00:00, 30.15it/s]


In [29]:
output

[Output(atac=TrackData(values=array([[3.3676624e-06],
        [9.3994141e-03],
        [5.1879883e-03],
        ...,
        [6.1035156e-03],
        [5.6457520e-04],
        [1.1657715e-02]], shape=(16384, 1), dtype=float32), metadata=                   name strand Assay title ontology_curie biosample_name  \
 0  EFO:0001086 ATAC-seq      .    ATAC-seq    EFO:0001086           A549   
 
   biosample_type biosample_life_stage data_source endedness  \
 0      cell_line                adult      encode    single   
 
    genetically_modified  nonzero_mean  
 0                 False      1.497697  , resolution=1, interval=None, uns=None), cage=None, dnase=None, rna_seq=None, chip_histone=None, chip_tf=None, splice_sites=None, splice_site_usage=None, splice_junctions=None, contact_maps=None, procap=None),
 Output(atac=TrackData(values=array([[3.1739473e-06],
        [9.0942383e-03],
        [5.0354004e-03],
        ...,
        [5.9204102e-03],
        [4.9591064e-04],
        [1.1352539e-

In [33]:
start=8042
end=8342
scores = []
for i, p in enumerate(output):
    atac = p.atac.values      
    score = atac[start:end].mean()
    scores.append(score)

print(scores)


[np.float32(0.12971967), np.float32(0.119610675), np.float32(0.092508994), np.float32(0.07714372), np.float32(0.04511631), np.float32(0.032938693), np.float32(0.026886154), np.float32(0.025635162), np.float32(0.027835343), np.float32(0.028688354), np.float32(0.03103461), np.float32(0.027437499)]


In [ ]:
os.environ["ALPHAGENOME_API_KEY"] = "xxxxxxxxxxxx"
dna_model = dna_client.create(os.environ["ALPHAGENOME_API_KEY"])

In [53]:
import pandas as pd

sequence = pd.read_csv('/hpc/home/rv103/igvf/revathy/GR-AP1/alphagenome/data/original-seq-16kb.txt', sep='\t', names=['ids','sequence'])
seq_list = list(sequence['sequence'])
ids = list(sequence['ids'])
seq_list


output = dna_model.predict_sequences(
    sequences=seq_list,
    requested_outputs={
        dna_client.OutputType.ATAC
    },
    ontology_terms=['EFO:0001086'],
)

print("Got predictions for", len(sequences), "sequences")

SEQ_LEN = 16384
CENTER_LEN = 300


start = (SEQ_LEN - CENTER_LEN) // 2
end = start + CENTER_LEN

scores = []
for i, p in enumerate(output):
    atac = p.atac.values      
    score = atac[start:end].mean()
    scores.append((ids[i], score))

scores_df = pd.DataFrame(scores, columns=['id','mean_score'])



100%|██████████| 12/12 [00:00<00:00, 28.83it/s]

Got predictions for 12 sequences


In [54]:
scores_df

,id,mean_score
0,seq1,0.129720
1,seq2,0.119611
2,seq3,0.092509
3,seq4,0.077144
4,seq5,0.045116
5,seq6,0.032939
6,seq7,0.026886
7,seq8,0.025635
8,seq9,0.027835
9,seq10,0.028688


In [35]:
import os
import re
import numpy as np
from alphagenome.models import dna_client
from alphagenome.data.ontology import OntologyTerm, OntologyType

TARGET_LEN = 16384  
CENTER_LEN = 300

def read_sequences_txt(path):
    seqs = []
    ids = []
    with open(path) as f:
        for line in f:
            if not line.strip() or line.startswith("#"):
                continue
            parts = re.split(r"\s+", line.strip())
            seq_id = parts[0]
            seq = parts[3].upper()
            if len(seq) != 300:
                raise ValueError("Expected 300bp sequences")
            seqs.append(seq)
            ids.append(seq_id)
    return ids, seqs


def pad_center_acgt(seq, target_len=TARGET_LEN, seed=0):
    rng = np.random.default_rng(seed)
    pad = target_len - len(seq)
    left = pad // 2
    right = pad - left
    return (
        "".join(rng.choice(list("ACGT"), left))
        + seq
        + "".join(rng.choice(list("ACGT"), right))
    )

txt_path = "/hpc/group/igvf/A549/GR-AP1/enhancer-seq/original-seq/original-seq.txt"
ids, seqs300 = read_sequences_txt(txt_path)

seqs16384 = [pad_center_acgt(s, seed=i) for i, s in enumerate(seqs300)]

os.environ["ALPHAGENOME_API_KEY"] = "xxxxxxxxxxxxx"
client = dna_client.create(os.environ["ALPHAGENOME_API_KEY"])

preds = client.predict_sequences(
    sequences=seqs16384,
    organism=dna_client.Organism.HOMO_SAPIENS,
    requested_outputs=[[*dna_client.OutputType][0]],
    ontology_terms=[], 
)


start = (TARGET_LEN - CENTER_LEN) // 2
end = start + CENTER_LEN

scores = []
for i, p in enumerate(preds):
    atac = p.atac.values       
    score = atac[start:end].mean()
    scores.append((ids[i], score))


scores.sort(key=lambda x: x[1], reverse=True)
for sid, s in scores:
    print(sid, s)


100%|██████████| 12/12 [00:01<00:00,  7.38it/s]

seq5 0.038800362
seq1 0.023042908
seq2 0.021674592
seq4 0.018305555
seq3 0.015902227
seq12 0.015762726
seq8 0.013532439
seq7 0.012797632
seq9 0.011844597
seq6 0.011659004
seq11 0.011526801
seq10 0.010706929


In [37]:
import pandas as pd
import numpy as np

rows = []

for seq_id, p in zip(ids, preds):
    atac = np.asarray(p.atac.values)   # shape: (L, tracks) or similar

    rows.append({
        "id": seq_id,
        "atac_mean": atac.mean(),
        "atac_max": atac.max(),
        "atac_std": atac.std(),
    })

df_atac = pd.DataFrame(rows)
df_atac


,id,atac_mean,atac_max,atac_std
0,seq1,0.024971,4.437500,0.053905
1,seq2,0.021250,1.718750,0.040695
2,seq3,0.018228,2.281250,0.036339
3,seq4,0.022951,5.656250,0.051800
4,seq5,0.027728,9.625000,0.071373
5,seq6,0.015502,1.062500,0.030171
6,seq7,0.020025,2.234375,0.042392
7,seq8,0.014980,1.515625,0.030007
8,seq9,0.016710,1.468750,0.033328
9,seq10,0.014760,1.171875,0.029234


In [47]:
print((preds[0].atac.values[0]))

[0.00521851 0.00209045 0.00372314 0.01055908 0.00245667 0.00247192
 0.00202942 0.00332642 0.00686646 0.00674438 0.00418091 0.00221252
 0.00445557 0.0020752  0.00878906 0.00891113 0.01745605 0.0123291
 0.01300049 0.01965332 0.01489258 0.00958252 0.0135498  0.01721191
 0.00805664 0.01068115 0.01135254 0.01635742 0.01092529 0.0098877
 0.01086426 0.01611328 0.01190186 0.01281738 0.01324463 0.01165771
 0.00921631 0.01782227 0.01660156 0.01293945 0.0145874  0.00778198
 0.01300049 0.01019287 0.01330566 0.0123291  0.01147461 0.00878906
 0.00994873 0.00765991 0.01660156 0.00805664 0.00823975 0.02111816
 0.0144043  0.00891113 0.00704956 0.0062561  0.00494385 0.00524902
 0.00585938 0.00247192 0.00799561 0.00379944 0.00386047 0.01586914
 0.00549316 0.0043335  0.00297546 0.00402832 0.01385498 0.00946045
 0.00744629 0.0123291  0.0144043  0.01196289 0.0168457  0.012146
 0.01434326 0.01031494 0.01074219 0.01037598 0.01647949 0.01672363
 0.01196289 0.01324463 0.00823975 0.01055908 0.01501465 0.01599121

In [11]:
from alphagenome.data.ontology import OntologyTerm, OntologyType

term = OntologyTerm(OntologyType.EFO, 1086)

In [14]:
print(OntologyType.EFO)

4


In [16]:
term.type

<OntologyType.EFO: 4>

In [17]:
hasattr(term, "to_proto")

True

In [28]:
dna_client.SUPPORTED_SEQUENCE_LENGTHS

{'SEQUENCE_LENGTH_2KB': 2048,
 'SEQUENCE_LENGTH_16KB': 16384,
 'SEQUENCE_LENGTH_100KB': 131072,
 'SEQUENCE_LENGTH_500KB': 524288,
 'SEQUENCE_LENGTH_1MB': 1048576}

In [29]:
list(dna_client.OutputType)


[ATAC,
 CAGE,
 DNASE,
 RNA_SEQ,
 CHIP_HISTONE,
 CHIP_TF,
 SPLICE_SITES,
 SPLICE_SITE_USAGE,
 SPLICE_JUNCTIONS,
 CONTACT_MAPS,
 PROCAP]